# 07 — Summary & Key Findings
## Tamil Nadu Assembly Elections (1971–2021)

**This notebook compiles the top findings from all analysis notebooks.**

In [ ]:
import sys
sys.path.insert(0, '/app')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from db import query, query_all, GENERAL_ELECTION_YEARS, POST_DELIM_YEARS, MAJOR_PARTIES, ALLIANCES

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

df = query_all()
ge = df[df.year.isin(GENERAL_ELECTION_YEARS)].copy()
ge['is_winner'] = ge.position == 1
winners = ge[ge.is_winner].copy()
recent = ge[ge.year >= 2011].copy()

## Key Finding 1: Perfect Anti-Incumbency Cycle
**Since 1996, TN has alternated power every single election** — the ruling party or alliance has never been re-elected. This is the strongest anti-incumbency pattern of any major Indian state.

In [ ]:
# Power alternation visualization
ruling_party = []
for year in GENERAL_ELECTION_YEARS:
    yr_winners = winners[winners.year == year]
    if len(yr_winners) == 0: continue
    top = yr_winners.party.value_counts()
    ruling_party.append({'year': year, 'party': top.index[0], 'seats': top.values[0], 'total': len(yr_winners)})

rp_df = pd.DataFrame(ruling_party)
rp_df['seat_pct'] = rp_df.seats / rp_df.total * 100

party_colors = {'DMK': '#E53935', 'ADMK': '#43A047', 'ADK': '#8E24AA', 'INC': '#1E88E5'}
fig, ax = plt.subplots(figsize=(16, 6))
colors = [party_colors.get(p, '#9E9E9E') for p in rp_df.party]
bars = ax.bar(rp_df.year, rp_df.seat_pct, color=colors, edgecolor='white', width=3)
ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5)
ax.set_title('Winning Party Seat Share — The Anti-Incumbency Cycle', fontsize=14)
ax.set_xlabel('Election Year')
ax.set_ylabel('Seat Share (%)')
for _, row in rp_df.iterrows():
    ax.annotate(f"{row.party}\n{row.seats}", (row.year, row.seat_pct),
               textcoords='offset points', xytext=(0, 5), ha='center', fontsize=8)

# Add legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=p) for p, c in party_colors.items()]
ax.legend(handles=legend_elements, loc='upper right')
plt.tight_layout()
plt.savefig('/app/output/07_anti_incumbency_cycle.png', dpi=150, bbox_inches='tight')
plt.show()

## Key Finding 2: FPTP Massively Amplifies Victory
**A party winning ~35-40% vote share can capture 55-65% of seats.** The Gallagher index shows TN elections are highly disproportional — small vote share swings produce massive seat swings.

## Key Finding 3: Rising Female Participation, Flat Win Rate
**Female candidate numbers are increasing** but their win rate remains ~5-8%, consistently lower than men (~8-12%). Parties field women in less winnable seats.

## Key Finding 4: Incumbency Is Dangerous
**Incumbent win rate in TN averages ~35-45%**, well below the national average. In wave elections (2011, 2021), incumbent survival drops to ~25-30%.

## Key Finding 5: Regional Blocs Are Distinct
- **Chennai City Region**: Most competitive, highest ENOP, lowest turnout
- **Western Region** (Salem, Coimbatore, Erode): ADMK stronghold, highest turnout
- **Southern Region** (Madurai+): Swing region, decides elections
- **Central Region** (Trichy, Thanjavur): DMK-leaning, agricultural

## Key Finding 6: Party System Is Bipolar
**ENOP has remained ~2.0-2.5** across all elections, confirming TN is a two-coalition system (DMK+ vs ADMK+). Third forces (DMDK in 2006, NTK in 2021) spike briefly but don't sustain.

In [ ]:
# Summary statistics dashboard
fig = plt.figure(figsize=(18, 14))
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.4, wspace=0.3)

# 1. Total records
ax0 = fig.add_subplot(gs[0, 0])
ax0.text(0.5, 0.5, f'{len(df):,}', fontsize=36, ha='center', va='center', fontweight='bold', color='#2196F3')
ax0.text(0.5, 0.2, 'Total Records', fontsize=14, ha='center', va='center', color='gray')
ax0.set_xlim(0, 1)
ax0.set_ylim(0, 1)
ax0.axis('off')

# 2. Years covered
ax1 = fig.add_subplot(gs[0, 1])
ax1.text(0.5, 0.5, f'{df.year.min()}–{df.year.max()}', fontsize=28, ha='center', va='center', fontweight='bold', color='#E53935')
ax1.text(0.5, 0.2, f'{len(GENERAL_ELECTION_YEARS)} General Elections', fontsize=14, ha='center', va='center', color='gray')
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 1)
ax1.axis('off')

# 3. Parties
ax2 = fig.add_subplot(gs[0, 2])
ax2.text(0.5, 0.5, f'{df.party.nunique()}', fontsize=36, ha='center', va='center', fontweight='bold', color='#43A047')
ax2.text(0.5, 0.2, 'Political Parties', fontsize=14, ha='center', va='center', color='gray')
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)
ax2.axis('off')

# 4. Turnout trend
ax3 = fig.add_subplot(gs[1, :])
turnout = ge.groupby('year').turnout_percentage.mean()
ax3.plot(turnout.index, turnout.values, 'o-', color='#2196F3', linewidth=2, markersize=8)
ax3.fill_between(turnout.index, turnout.values, alpha=0.1, color='#2196F3')
ax3.set_title('Average Voter Turnout Over Time', fontsize=13)
ax3.set_ylabel('Turnout (%)')

# 5. Seat count by major party
ax4 = fig.add_subplot(gs[2, :])
for party, color in [('DMK', '#E53935'), ('ADMK', '#43A047'), ('INC', '#1E88E5')]:
    seats = winners[winners.party == party].groupby('year').size()
    ax4.plot(seats.index, seats.values, 'o-', label=party, color=color, linewidth=2)
ax4.set_title('Seats Won by Major Parties', fontsize=13)
ax4.set_ylabel('Seats')
ax4.legend()

plt.savefig('/app/output/07_summary_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

## Key Finding 7: Education and Professional Background
- **Graduate-level candidates have the highest win rate** (~12-15%) among education categories
- **Business owners and lawyers** are overrepresented among winners vs the general candidate pool
- Education data availability improves significantly from 2011 onwards (MyNeta data)

## Key Finding 8: Turncoats Fare Poorly
Party-switchers (turncoats) consistently have a **lower win rate** than candidates who remain loyal to their party. Voters appear to punish defection.

## Key Finding 9: Frivolous Candidacy Is Increasing
The **percentage of candidates losing their deposit** has risen from ~60% in earlier elections to ~75%+ in 2016-2021, driven by party proliferation (307 parties in the dataset). The average number of candidates per constituency has also risen to ~18.

## Key Finding 10: Predictive Model Insights
The XGBoost classifier's top predictive features for winning are:
1. **is_major_party** — Candidates from major parties are disproportionately likely to win
2. **prev_party_avg_vote_share** — Historical party strength matters
3. **n_cand / ENOP** — More fragmented races change dynamics
4. **is_incumbent_party** — Being the incumbent party (not individual) is predictive
5. **turnout_percentage** — Higher turnout correlates with anti-incumbency

In [ ]:
# 2026 Outlook
print('=== 2026 ELECTION OUTLOOK (Based on Historical Patterns) ===')
print()
print('Historical Pattern: Power has alternated every election since 1996.')
print('2021 Winner: DMK (133 seats)')
print()
print('If anti-incumbency pattern holds → ADMK+ alliance favored.')
print()
print('Key factors to watch:')
print('  1. Alliance configurations (especially PMK, NTK positioning)')
print('  2. ADMK leadership succession (post-Jayalalithaa era performance)')
print('  3. DMK governance record and welfare scheme impact')
print('  4. Third force consolidation (NTK got 6.5% vote share in 2021)')
print('  5. BJP expansion attempt (currently <3% vote share)')